# 📊 Phase 0: Data Preprocessing & Cleaning

This notebook performs data preprocessing, cleaning, feature engineering, encoding, normalization, and visualization.

In [2]:

# Step Processing Module

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# قراءة البيانات
df = pd.read_csv(r'd:\courses\Artificial Intelligence\AI Project\student_exam_data_new.csv')

print("="*60)
print("📊 PHASE 0: DATA PREPROCESSING & CLEANING")
print("="*60)


ModuleNotFoundError: No module named 'numpy'

## 1️⃣ Initial Data Overview

In [ ]:

print("1. Initial Data Overview:")
print(f"   • Shape: {df.shape}")
print(f"   • Columns: {list(df.columns)}")
print(f"   • Data Types:\n{df.dtypes}")
print("-"*40)


## 2️⃣ Missing Values Analysis

In [ ]:

print("2. Missing Values Analysis:")
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percent
})
print(missing_df)
print("-"*40)


## 3️⃣ Duplicate Rows Check

In [1]:

print("3. Duplicate Rows Check:")
duplicates = df.duplicated().sum()
print(f"   • Total duplicate rows: {duplicates}")
if duplicates > 0:
    print(f"   • Removing {duplicates} duplicate rows...")
    df = df.drop_duplicates()
    print(f"   • New shape after removal: {df.shape}")
print("-"*40)


3. Duplicate Rows Check:


NameError: name 'df' is not defined

<!-- Delete Duplicate Date  -->
print("Delete Duplicate ")
if duplicate_count > 0:
    df_clean = df.drop_duplicates()
    print(f"   has been deleted {duplicate_count} Duplicate date")
    print(f"  After Delete: {df_clean.shape[0]}")

## 4️⃣ Data Integrity Check

In [ ]:

print("4. Data Integrity Check:")
print("   • Study Hours Range:")
print(f"     Min: {df['Study Hours'].min():.2f}, Max: {df['Study Hours'].max():.2f}")
print(f"     Study Hours > 24 (Unreasonable): {(df['Study Hours'] > 24).sum()} rows")
print(f"     Study Hours = 0: {(df['Study Hours'] == 0).sum()} rows")

print("\n   • Exam Scores Range:")
print(f"     Min: {df['Previous Exam Score'].min():.2f}, Max: {df['Previous Exam Score'].max():.2f}")
print(f"     Scores < 0: {(df['Previous Exam Score'] < 0).sum()} rows")
print(f"     Scores > 100: {(df['Previous Exam Score'] > 100).sum()} rows")

print("\n   • Pass/Fail Values:")
print(f"     Unique values: {df['Pass/Fail'].unique()}")
print(f"     Invalid values (not 0 or 1): {((df['Pass/Fail'] != 0) & (df['Pass/Fail'] != 1)).sum()}")
print("-"*40)


## 5️⃣ Outliers Detection & Handling

In [ ]:

Q1_hours = df['Study Hours'].quantile(0.25)
Q3_hours = df['Study Hours'].quantile(0.75)
IQR_hours = Q3_hours - Q1_hours
lower_hours = Q1_hours - 1.5 * IQR_hours
upper_hours = Q3_hours + 1.5 * IQR_hours

df['Study Hours Cleaned'] = df['Study Hours'].clip(lower=lower_hours, upper=upper_hours)

Q1_scores = df['Previous Exam Score'].quantile(0.25)
Q3_scores = df['Previous Exam Score'].quantile(0.75)
IQR_scores = Q3_scores - Q1_scores
lower_scores = Q1_scores - 1.5 * IQR_scores
upper_scores = Q3_scores + 1.5 * IQR_scores

df['Previous Exam Score Cleaned'] = df['Previous Exam Score'].clip(lower=lower_scores, upper=upper_scores)

print("Outlier handling applied using capping (Winsorization).")


## 6️⃣ Feature Engineering

In [ ]:

def categorize_study_hours(hours):
    if hours < 3:
        return 'Very Low'
    elif hours < 6:
        return 'Low'
    elif hours < 8:
        return 'Medium'
    elif hours < 10:
        return 'High'
    else:
        return 'Very High'

def categorize_scores(score):
    if score < 50:
        return 'Fail'
    elif score < 65:
        return 'Pass'
    elif score < 80:
        return 'Good'
    elif score < 90:
        return 'Very Good'
    else:
        return 'Excellent'

df['Study_to_Score_Ratio'] = df['Study Hours'] / (df['Previous Exam Score'] + 0.0001)
df['Study_Category'] = df['Study Hours'].apply(categorize_study_hours)
df['Score_Category'] = df['Previous Exam Score'].apply(categorize_scores)

df[['Study_Category', 'Score_Category']].head()


## 7️⃣ Encoding

In [ ]:

le_study = LabelEncoder()
le_score = LabelEncoder()

df['Study_Category_Encoded'] = le_study.fit_transform(df['Study_Category'])
df['Score_Category_Encoded'] = le_score.fit_transform(df['Score_Category'])

le_study.classes_, le_score.classes_


## 8️⃣ Normalization

In [ ]:

def min_max_scaler(series):
    return (series - series.min()) / (series.max() - series.min())

df['Study Hours Normalized'] = min_max_scaler(df['Study Hours Cleaned'])
df['Exam Score Normalized'] = min_max_scaler(df['Previous Exam Score Cleaned'])

df[['Study Hours Normalized', 'Exam Score Normalized']].describe()


## 🔟 Visualization

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(df['Study Hours'], bins=20, alpha=0.5, label='Original')
axes[0, 0].hist(df['Study Hours Cleaned'], bins=20, alpha=0.5, label='Cleaned')
axes[0, 0].legend()

axes[0, 1].hist(df['Previous Exam Score'], bins=20, alpha=0.5, label='Original')
axes[0, 1].hist(df['Previous Exam Score Cleaned'], bins=20, alpha=0.5, label='Cleaned')
axes[0, 1].legend()

scatter = axes[1, 0].scatter(df['Study Hours Cleaned'], df['Previous Exam Score Cleaned'], c=df['Pass/Fail'])
plt.colorbar(scatter, ax=axes[1, 0])

study_counts = df['Study_Category'].value_counts()
axes[1, 1].bar(study_counts.index, study_counts.values)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
